# Dictionary-Based Sentiment Analysis
Lexicon approach using the Loughran-McDonald Master Dictionary. Runs evaluation and logs results to MLflow.

In [ ]:
import sys
sys.path.insert(0, "..")

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dictionary.scorer import load_dictionary, predict
from src.utils.metrics import compute_metrics


In [ ]:
# Run evaluation on data.csv — logs to MLflow
import subprocess
result = subprocess.run(
    [sys.executable, "../src/dictionary/evaluate.py", "--dataset", "../data/data.csv", "--threshold", "0.3"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


In [ ]:
# Run evaluation on all-data.csv
result2 = subprocess.run(
    [sys.executable, "../src/dictionary/evaluate.py", "--dataset", "../data/all-data.csv", "--threshold", "0.3"],
    capture_output=True, text=True
)
print(result2.stdout)
if result2.returncode != 0:
    print(result2.stderr)


In [ ]:
# Load latest dictionary run from MLflow
mlflow.set_tracking_uri("../mlruns")
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("financial-sentiment")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="params.approach = 'dictionary'",
    order_by=["start_time DESC"],
    max_results=1,
)
run = runs[0]
print("Params:", run.data.params)
print("Metrics:", {k: round(v, 4) for k, v in run.data.metrics.items()})


In [ ]:
# Compare all approaches side by side
all_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)
comparison = pd.DataFrame([
    {
        "approach": r.data.params.get("approach"),
        "dataset": r.data.params.get("dataset"),
        "accuracy": round(r.data.metrics.get("accuracy", 0), 4),
        "f1_macro": round(r.data.metrics.get("f1_macro", 0), 4),
    }
    for r in all_runs
])
comparison
